# Part 2 on Colab: the ML extension (full model set)

This runs **our ML extension** of the StudentLife reproduction (it is *not* part of the original paper):
predicting each student's next-day mood (PAM positive affect) from last night's sleep, yesterday's
activity / sociability / phone darkness and their recent mood, with ridge regression, random forest
and gradient boosting against simple baselines.

Why Colab: on the laptop this was built on, Windows Smart App Control blocks scikit-learn's tree-model
files, so only ridge regression runs there. Colab has no such block.

**Steps**
1. On your computer: `python mood_research/make_colab_bundle.py` -> creates `colab_bundle.zip`.
2. Run the cells below in order (Runtime -> Run all). Upload `colab_bundle.zip` when asked.
3. The last cell downloads `ml_results_colab.zip` - unzip it into `mood_research/` to replace the
   ridge-only results with the full ones.

In [ ]:
# 1. upload colab_bundle.zip
from google.colab import files
import zipfile, os
up = files.upload()
name = next(iter(up))
zipfile.ZipFile(name).extractall("/content")
print(sorted(os.listdir("/content/mood_research")))

In [ ]:
# 2. the one package Colab does not have (reads the dataset's R files)
!pip -q install pyreadr
import sklearn, statsmodels, pandas
print("scikit-learn", sklearn.__version__, "| statsmodels", statsmodels.__version__, "| pandas", pandas.__version__)

If `colab_bundle.zip` did not include `data/processed/daily.pkl`, the next cell rebuilds it from the raw
data (downloads 230 MB from Zenodo, about 10 minutes). Otherwise it just loads it.

In [ ]:
# 3. data
%cd /content/mood_research
import os
if not os.path.exists("data/processed/daily.pkl"):
    !python get_data.py
    !cd src && python -c "from features import build_daily; build_daily()"
else:
    print("using the bundled daily feature table")

In [ ]:
# 4. run the ML extension (about 5-15 minutes)
%cd /content/mood_research/src
!python -W ignore extension.py

In [ ]:
# 5. the figures
from IPython.display import Image, display
for f in ["fig4_model_comparison", "fig5_predicted_vs_actual", "fig6_permutation_importance",
          "fig7_mixed_model_associations", "fig8_error_analysis"]:
    print(f); display(Image(f"/content/mood_research/figures/{f}.png"))

In [ ]:
# 6. download the results to put back into mood_research/
import shutil
%cd /content/mood_research
!zip -q -r /content/ml_results_colab.zip results/ml_* results/mixed_model_associations.csv figures/fig4* figures/fig5* figures/fig6* figures/fig7* figures/fig8*
files.download("/content/ml_results_colab.zip")